In [2]:
import pandas as pd
from rich import print

In [3]:
df = pd.read_csv(filepath_or_buffer="./data.csv")

### 0. Check df. Little EDA

In [4]:
df.head()

,project_type,project,job,question,assessor,dt,label
0,project_type_1,project_1,job_14008,question_3,assessor_53,2025-07-08,1
1,project_type_0,project_0,job_14004,question_4,assessor_55,2025-07-08,1
2,project_type_1,project_1,job_14020,question_2,assessor_14,2025-07-08,0
3,project_type_1,project_1,job_14021,question_0,assessor_6,2025-07-08,1
4,project_type_0,project_2,job_14032,question_2,assessor_47,2025-07-08,0


In [5]:
df.shape

(708, 7)

In [6]:
df.dtypes

project_type      str
project           str
job               str
question          str
assessor          str
dt                str
label           int64
dtype: object

In [7]:
class ColNms:
    PROJECT_TYPE = "project_type"
    PROJECT = "project"
    JOB = "job"
    QUESTION = "question"
    ASSESSOR = "assessor"
    DT = "dt"
    LABEL = "label"

In [8]:
df[ColNms.DT].dtype

<StringDtype(storage='python', na_value=nan)>

In [9]:
df.isna().sum()

project_type    0
project         0
job             0
question        0
assessor        0
dt              1
label           0
dtype: int64

In [10]:
df.duplicated().sum()

np.int64(1)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   project_type  708 non-null    str  
 1   project       708 non-null    str  
 2   job           708 non-null    str  
 3   question      708 non-null    str  
 4   assessor      708 non-null    str  
 5   dt            707 non-null    str  
 6   label         708 non-null    int64
dtypes: int64(1), str(6)
memory usage: 38.8 KB


In [12]:
df.describe()

,label
count,708.000000
mean,0.549435
std,0.497902
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


### 1. Fix the dtypes

In [13]:
df[ColNms.DT] = pd.to_datetime(df[ColNms.DT])

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   project_type  708 non-null    str           
 1   project       708 non-null    str           
 2   job           708 non-null    str           
 3   question      708 non-null    str           
 4   assessor      708 non-null    str           
 5   dt            707 non-null    datetime64[us]
 6   label         708 non-null    int64         
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 38.8 KB


### 2. Clean edge cases

In [15]:
df = df.dropna()

In [16]:
df.shape

(707, 7)

In [17]:
df = df.drop_duplicates()

In [18]:
df.shape

(706, 7)

We dropped the dupes which are exactly the same but there is a subtler case:

since we have a composite PK there might me the case that the same assessor labeled the same question twice with different labels.

In [19]:
pk = [ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION, ColNms.ASSESSOR]

In [20]:
conflicts = df.groupby(pk)[ColNms.LABEL]

In [21]:
conflicts.head()

0      1
1      1
2      0
3      1
4      0
      ..
703    0
704    0
705    1
706    1
707    1
Name: label, Length: 706, dtype: int64

In [22]:
conflicts = df.groupby(pk)[ColNms.LABEL].nunique()

In [23]:
conflicts.head(100)

project    job        question    assessor   
project_0  job_14000  question_0  assessor_24    1
                                  assessor_41    1
                                  assessor_49    1
                                  assessor_5     1
                                  assessor_56    1
                                                ..
           job_14004  question_1  assessor_10    1
                                  assessor_44    1
                                  assessor_46    1
                                  assessor_50    1
                                  assessor_8     1
Name: label, Length: 100, dtype: int64

In [24]:
conflicts[conflicts > 1]

Series([], Name: label, dtype: int64)

### 3. Q1: prepare the df in the format other teams can consume

In [25]:
df[ColNms.PROJECT].nunique()

3

how many questions per project?

In [26]:
df.groupby(ColNms.PROJECT)[ColNms.QUESTION].nunique()

project
project_0    5
project_1    5
project_2    5
Name: question, dtype: int64

how many assessors labeled the same question?

In [27]:
with pd.option_context("display.max_rows", None):
    df_questions = df.groupby([ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION])[
        ColNms.ASSESSOR
    ].nunique()
    print(df_questions[df_questions > 5])

project    job        question  
project_1  job_14010  question_1    6
Name: assessor, dtype: int64

In [28]:
df_questions.describe()

count    141.000000
mean       5.007092
std        0.084215
min        5.000000
25%        5.000000
50%        5.000000
75%        5.000000
max        6.000000
Name: assessor, dtype: float64

we see it is a majority vote. let's prepare a df with lables based on majority vote

In [29]:
votes = df.groupby([ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION])[ColNms.LABEL].mean()

In [30]:
majority = votes[
    votes >= 0.5
]  # here you failed: you selected the rows whose mean is more or = 0.5. you filtered out.

In [31]:
majority.head()

project    job        question  
project_0  job_14000  question_0    0.8
                      question_2    0.8
           job_14001  question_2    1.0
                      question_4    0.6
           job_14002  question_1    1.0
Name: label, dtype: float64

In [32]:
majority = votes >= 0.5

In [33]:
majority.head()

project    job        question  
project_0  job_14000  question_0     True
                      question_1    False
                      question_2     True
                      question_3    False
                      question_4    False
Name: label, dtype: bool

In [34]:
majority = (votes >= 0.5).astype(int)

In [35]:
majority.head()

project    job        question  
project_0  job_14000  question_0    1
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    0
Name: label, dtype: int64

In [36]:
with pd.option_context("display.max_rows", None):
    print(majority)

project    job        question  
project_0  job_14000  question_0    1
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    0
           job_14001  question_0    0
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    1
           job_14002  question_0    0
                      question_1    1
                      question_2    1
                      question_3    1
           job_14003  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
           job_14004  question_0    0
                      question_1    0
                      question_2    0
                      question_3    1
                      question_4    0
           job_14005  question_0    1
                      question_1    0
                      question_2    0
           job_14006  question_0    0
                      question_1    1
                      question_2    0
           job_14007  question_0    1
                      question_1    1
                      question_2    0
                      question_3    1
                      question_4    0
project_1  job_14008  question_0    1
                      question_1    0
                      question_2    1
                      question_3    1
                      question_4    1
           job_14009  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
           job_14010  question_0    0
                      question_1    0
                      question_2    1
           job_14011  question_0    0
                      question_1    1
                      question_2    0
           job_14012  question_0    1
                      question_1    1
                      question_2    0
                      question_3    0
           job_14013  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
           job_14014  question_0    1
                      question_1    0
                      question_2    1
           job_14015  question_0    1
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    1
           job_14016  question_0    0
                      question_1    0
                      question_2    1
                      question_3    1
           job_14017  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
                      question_4    0
           job_14018  question_0    0
                      question_1    0
                      question_2    0
                      question_3    1
           job_14019  question_0    1
                      question_1    1
                      question_2    1
                      question_3    0
           job_14020  question_0    1
                      question_1    1
                      question_2    0
           job_14021  question_0    1
                      question_1    0
                      question_2    1
                      question_3    1
                      question_4    0
project_2  job_14022  question_0    0
                      question_1    0
                      question_2    0
                      question_3    1
                      question_4    0
           job_14023  question_0    0
                      question_1    1
                      question_2    1
                      question_3    0
           job_14024  question_0    1
                      question_1    1
                      question_2    1
                      question_3    0
                      question_4    0
           job_

In [37]:
majority.reset_index()

,project,job,question,label
0,project_0,job_14000,question_0,1
1,project_0,job_14000,question_1,0
2,project_0,job_14000,question_2,1
3,project_0,job_14000,question_3,0
4,project_0,job_14000,question_4,0
...,...,...,...,...
136,project_2,job_14033,question_2,1
137,project_2,job_14033,question_3,1
138,project_2,job_14034,question_0,1
139,project_2,job_14034,question_1,0


In [38]:
majority.reset_index(name="majority_vote")

,project,job,question,majority_vote
0,project_0,job_14000,question_0,1
1,project_0,job_14000,question_1,0
2,project_0,job_14000,question_2,1
3,project_0,job_14000,question_3,0
4,project_0,job_14000,question_4,0
...,...,...,...,...
136,project_2,job_14033,question_2,1
137,project_2,job_14033,question_3,1
138,project_2,job_14034,question_0,1
139,project_2,job_14034,question_1,0


### 4. Assessors quality label

In [39]:
g = df.groupby([ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION])[ColNms.LABEL]

In [ ]:
g.head()  # g is just a groupby object and .head() behaves weirdly on it

0      1
1      1
2      0
3      1
4      0
      ..
703    0
704    0
705    1
706    1
707    1
Name: label, Length: 705, dtype: int64

In [44]:
g.dtype

project    job        question  
project_0  job_14000  question_0    int64
                      question_1    int64
                      question_2    int64
                      question_3    int64
                      question_4    int64
                                    ...  
project_2  job_14033  question_2    int64
                      question_3    int64
           job_14034  question_0    int64
                      question_1    int64
                      question_2    int64
Name: label, Length: 141, dtype: object

In [56]:
# we can check the first group
g.get_group(("project_0", "job_14000", "question_0"))  # .reset_index(drop=True)

80     1
227    1
263    1
336    0
403    1
Name: label, dtype: int64

In [53]:
g.size()  # a really useful method to check how we grouped

project    job        question  
project_0  job_14000  question_0    5
                      question_1    5
                      question_2    5
                      question_3    5
                      question_4    5
                                   ..
project_2  job_14033  question_2    5
                      question_3    5
           job_14034  question_0    5
                      question_1    5
                      question_2    5
Name: label, Length: 141, dtype: int64

In [ ]:
df["grp_sum"] = g.transform("sum")  # the number of ones
df.head()

,project_type,project,job,question,assessor,dt,label,grp_sum
0,project_type_1,project_1,job_14008,question_3,assessor_53,2025-07-08,1,5
1,project_type_0,project_0,job_14004,question_4,assessor_55,2025-07-08,1,2
2,project_type_1,project_1,job_14020,question_2,assessor_14,2025-07-08,0,0
3,project_type_1,project_1,job_14021,question_0,assessor_6,2025-07-08,1,3
4,project_type_0,project_2,job_14032,question_2,assessor_47,2025-07-08,0,1


In [49]:
df.shape

(706, 8)

In [58]:
df.iloc[0]

project_type         project_type_1
project                   project_1
job                       job_14008
question                 question_3
assessor                assessor_53
dt              2025-07-08 00:00:00
label                             1
grp_sum                           5
Name: 0, dtype: object

In [ ]:
df["grp_n"] = g.transform("count")  # the number of labels
df.head()

,project_type,project,job,question,assessor,dt,label,grp_sum,grp_n
0,project_type_1,project_1,job_14008,question_3,assessor_53,2025-07-08,1,5,5
1,project_type_0,project_0,job_14004,question_4,assessor_55,2025-07-08,1,2,5
2,project_type_1,project_1,job_14020,question_2,assessor_14,2025-07-08,0,0,5
3,project_type_1,project_1,job_14021,question_0,assessor_6,2025-07-08,1,3,5
4,project_type_0,project_2,job_14032,question_2,assessor_47,2025-07-08,0,1,5


In [ ]:
df = df[df["grp_n"] > 1]  # drop questions with only one assessor
# since there is no peer to compare against

In [60]:
df["peer_mean"] = (df["grp_sum"] - df["label"]) / (df["grp_n"] - 1)
# well if 5 assessors have 5 ones we have mean 1
# of 4 out of 5 then mean is 4/5= 0.8
# if we want leave one out then we need to substract the label
# (if it is 0 it doesn't contribute to the overall sum either way)
# and we devide by the number of assesors - 1 which will be the mean of the peers for each row

In [61]:
df["dev"] = (
    df["label"] - df["peer_mean"]
)  # how you deviate from the peers in absolute terms

In [62]:
quality = (
    df.groupby("assessor")["dev"]
    .agg(bias="median", noise="std", n_replies="count")
    .sort_values("noise", ascending=False)  # worst at top
)
quality.head(10)

,bias,noise,n_replies
assessor,,,
assessor_47,0.000,0.620667,11
assessor_25,0.000,0.602919,17
assessor_27,0.000,0.571083,11
assessor_36,-0.625,0.570088,6
assessor_54,0.000,0.552591,15
assessor_44,0.000,0.533785,17
assessor_39,0.000,0.520526,14
assessor_51,0.000,0.498395,13
assessor_0,0.250,0.475073,9


In [64]:
quality["noise"] = round(quality["noise"] * 100, 1)

In [69]:
quality.head(100)

,bias,noise,n_replies
assessor,,,
assessor_47,0.000,62.1,11
assessor_25,0.000,60.3,17
assessor_27,0.000,57.1,11
assessor_36,-0.625,57.0,6
assessor_54,0.000,55.3,15
assessor_44,0.000,53.4,17
assessor_39,0.000,52.1,14
assessor_51,0.000,49.8,13
assessor_0,0.250,47.5,9


In [68]:
df.groupby("assessor")["dev"].size()

assessor
assessor_0      9
assessor_1      8
assessor_10    16
assessor_11     9
assessor_12     7
assessor_13    15
assessor_14     9
assessor_15    13
assessor_16     8
assessor_17    14
assessor_18    15
assessor_19     8
assessor_2     16
assessor_20    13
assessor_21     8
assessor_22    12
assessor_23    15
assessor_24    12
assessor_25    17
assessor_26    12
assessor_27    11
assessor_28    17
assessor_29    11
assessor_3     10
assessor_30     8
assessor_31    14
assessor_32    14
assessor_33    12
assessor_34    12
assessor_35    11
assessor_36     6
assessor_37     7
assessor_38     5
assessor_39    14
assessor_4     12
assessor_40     8
assessor_41    12
assessor_42    10
assessor_43    11
assessor_44    17
assessor_45    15
assessor_46    14
assessor_47    11
assessor_48    16
assessor_49    10
assessor_5     12
assessor_50    12
assessor_51    13
assessor_52     7
assessor_53    18
assessor_54    15
assessor_55    12
assessor_56    16
assessor_57    12
assessor_58     9
a

You're right, that was too compressed. Here's the fuller version with the mechanics you actually worked through:

**The data and the goal.** One row per assessor's answer to a question. A question is uniquely identified by `(project, job, question)` — never `question` alone, since names repeat across jobs. You want to score each assessor by how much they disagree with their peers, which means for every reply you compare their label against what *everyone else* said on that same question.

**Step 1 — peer mean, with the leave-one-out trick.** For each question you have the group sum (count of 1s, since labels are binary) and group count (number of assessors). The mean of everyone *except* the current assessor is the closed form:

```
peer_mean = (grp_sum - label) / (grp_n - 1)
```

The exclusion is mathematical, not a filter — each row subtracts its own label from the group total, which *is* "everyone but me." No assessor ID needed. For a `label=0` row the numerator is unchanged (they added nothing to the sum of 1s) but the denominator still drops by 1 (one fewer peer) — that asymmetry is why it works cleanly for binary labels. Computed with `transform`, which broadcasts the group value back to every row in one vectorized pass instead of an O(n²) loop.

**Step 2 — deviation.** `dev = label - peer_mean`, a continuous value in [-1, 1]. Note `peer_mean` is *not* 0/1 — with 4 peers it can be 0, 0.25, 0.5, 0.75, 1, so `dev` has real granularity.

- sign → direction of lean: positive = he sat higher than peers (more toward 1), negative = lower (more toward 0)
- magnitude → how much he disagreed: `|dev|` near 0 = agreement, near 1 = strong disagreement
- agreement is about magnitude, **not** the 0.5 threshold — that was only for the majority-vote step

Crucially this is per-question: you get one `dev` for each question the assessor touched.

**Step 3 — aggregate across all his questions.** Now group by assessor and collapse his list of devs into two numbers:

- **median (bias)** — his typical lean. ≈0 = balanced with consensus, ≠0 = systematically skewed one way. Median over mean because it's robust to one-off weird/ambiguous questions; with enough replies the two converge anyway, but median is the safer default and that was your motivation.
- **std (noise)** — consistency of his deviation *around his own typical behavior*. Low = he deviates predictably, high = erratic, swinging question to question. It measures spread, not magnitude — an assessor always wrong by exactly -1 has a big average deviation but **zero** std, because he's perfectly consistent.

**Why both metrics.** A median near 0 looks like agreement but can hide chaos — deviations of +1 and -1 cancel to a median of 0. Only a low std confirms the agreement is real. So:

```
median≈0, std≈0    → reliably agrees with peers          → good assessor
median≠0, std≈0    → consistently leans one way           → biased but predictable/correctable
median≈0, std high → swings randomly, cancels to zero     → noisy, unreliable
median≠0, std high → biased AND erratic                   → worst case
```

Std is the headline "bigger = worse" score; median diagnoses the *kind* of badness. Drop questions with only one assessor first — no peers means no leave-one-out possible.

**Std (noise) is the primary quality metric** — that's what your original brief specified, and it's the right call.

Here's the reasoning:

**Std catches the genuinely problematic assessors.** An erratic assessor — sometimes agreeing, sometimes wildly off, no pattern — is the worst kind because you can't trust or correct them. High std flags exactly this. Bigger std = worse = the headline score you sort by.

**Bias (median) alone is misleading and not directly usable as a quality score.** Two reasons:

1. A consistent bias is often *correctable*. If someone systematically leans toward 1, you can adjust for it. That's not great, but it's predictable — less damaging than randomness.

2. Bias can be near 0 while the assessor is terrible, because opposite deviations cancel out. So a low bias does not mean a good assessor.

**But you report both.** Bias is a *diagnostic*, not the score. It tells you *how* a bad assessor is bad once std has flagged them:

```
sort by std descending → find your worst assessors
then look at their bias → understand the failure mode
```

So the answer to "which metric": **rank by std, use bias to interpret.** If you were forced to pick a single number to threshold on for "remove this assessor," it's std.

One caveat worth stating: weight by `n_replies` before acting. An assessor with std=0.6 over 5 replies is much less certain than one with std=0.6 over 200 replies. Filter out low-sample assessors before ranking, or the top of your "worst" list will just be small-sample noise.

## Alternatives

Far from the only way — this is one reasonable approach, and not even the most principled one. Here are the real alternatives, roughly from simplest to most rigorous:

**1. Raw agreement rate.** Simplest possible: what fraction of the time does the assessor match the majority vote? One number, trivially interpretable, no leave-one-out math. Weakness: treats a 3-2 split the same as 5-0, and ignores question difficulty.

**2. Cohen's kappa.** Agreement *corrected for chance*. If labels are 90% "1", an assessor who blindly always says "1" gets 90% raw agreement but adds nothing — kappa strips that out. This is the classic textbook metric for two raters. For more than two, **Fleiss' kappa** or **Krippendorff's alpha** generalize it. These are what an academic or a careful annotation team would reach for first.

**3. Confusion-matrix metrics vs majority.** Treat majority vote as ground truth and compute per-assessor precision, recall, F1. Useful when false positives and false negatives matter differently (fraud labeling, medical) — your std metric is symmetric and can't capture that asymmetry.

**4. Dawid-Skene (the "proper" answer).** This is the gold standard for exactly your problem. It's an EM algorithm that **jointly estimates** the true label of each question *and* each assessor's reliability (their full confusion matrix), without assuming the majority is correct. Your method has a known weakness — it treats majority as truth, so a careful assessor who breaks from a sloppy majority looks "noisy." Dawid-Skene fixes that by letting reliable assessors *outvote* unreliable ones iteratively. If a Revolut interviewer asked "what's wrong with your approach," the answer is "it trusts the majority; Dawid-Skene doesn't."

**5. Difficulty-aware models (item-response theory / GLAD).** Go further and model that some *questions* are inherently hard. An assessor who misses only hard questions is better than one who misses easy ones. IRT-style models separate assessor skill from question difficulty.

So where does your std method sit? It's a **reasonable, fast, interpretable heuristic** — good for a first pass or a live interview where you can't implement EM on a whiteboard. But it makes two assumptions worth naming out loud:

- majority = truth (Dawid-Skene drops this)
- all questions equally hard (IRT/GLAD drop this)

The strong interview move is exactly what you'd now say: "I'd use leave-one-out deviation as a quick, vectorizable heuristic, flag that it assumes the majority is correct, and mention Dawid-Skene as the principled version if we have time and the stakes justify it." That demonstrates you know both the pragmatic answer *and* the textbook one — which is what separates a senior answer from a junior one.

No — I wouldn't say it's the *best*. It's a good *pragmatic* approach, but "best" depends on what you're optimizing for.

If I had to rank them by what they're actually best *for*:

**Best for correctness/rigor → Dawid-Skene.** It doesn't assume the majority is right, it jointly estimates true labels and assessor reliability. If the stakes are real (you're going to fire assessors, or the labels train a production model), this is the defensible choice. The std method's core weakness — trusting the majority — is exactly what Dawid-Skene fixes.

**Best for a live interview / quick first pass → your std + LOO method.** Fast, vectorizable, explainable in two minutes, no EM convergence to babysit. This is where it shines, and it's why it was the right thing to build under interview pressure.

**Best for interpretability to non-technical stakeholders → raw agreement rate or kappa.** "This assessor agrees with the group 72% of the time" is something a manager understands instantly. Std-of-deviations is not.

So the honest answer is: **the std/LOO method is the best *tradeoff* of speed, simplicity, and signal — but not the most *correct* method.** Calling any single one "the best" without specifying the constraint is the trap.

In a real production setting at somewhere like Revolut, I'd actually use std/LOO as a **cheap pre-filter** to flag suspicious assessors fast, then run Dawid-Skene on the flagged set to confirm before acting. Pragmatic screen, rigorous confirmation. That layered answer is stronger than committing to any one method.